In [ ]:
Sys.time()

In [ ]:
suppressPackageStartupMessages(library(ArchR))
suppressPackageStartupMessages(library(ggplot2))
suppressPackageStartupMessages(library(tidyverse))
suppressPackageStartupMessages(library(data.table))
suppressPackageStartupMessages(library(BSgenome))
suppressPackageStartupMessages(library(BSgenome.Hsapiens.UCSC.hg38))

In [ ]:
addArchRGenome("hg38")
addArchRThreads(threads = 8)

projdir   <- '/u/project/cluo/terencew/igvf/2023_YR2/multiome/'
proj_path <- paste0(projdir, 'atac/archr/processed/YS3_intersect_rna')
out_dir   <- paste0(projdir, 'csv/footprint/')

clusters_order <- c('Start', 'Fail1', 'Inter1', 'Inter2', 'Fail2', 'IPS',
                    'Sendai', 'Delayed', 'Inflamed', 'Skin')

## 1. Check project prerequisites

In [ ]:
proj <- loadArchRProject(path = proj_path, force = FALSE, showLogo = FALSE)
proj

In [ ]:
cat('Cell metadata columns:\n')
print(names(proj@cellColData))

cat('\nAvailable matrices:\n')
print(getAvailableMatrices(proj))

cat('\nGroupCoverages computed for:\n')
print(list.files(file.path(proj_path, 'GroupCoverages')))

In [ ]:
# Check if coarse_v7 GroupCoverages and motif annotations exist
has_coarse_v7_cov <- 'coarse_v7' %in% list.files(file.path(proj_path, 'GroupCoverages'))
has_motif         <- 'Motif' %in% names(proj@peakAnnotation)
has_coarse_v7_col <- 'coarse_v7' %in% names(proj@cellColData)

cat('coarse_v7 GroupCoverages:', has_coarse_v7_cov, '\n')
cat('Motif annotations:       ', has_motif, '\n')
cat('coarse_v7 cell metadata: ', has_coarse_v7_col, '\n')

if (!has_coarse_v7_cov || !has_motif) {
    cat('\nRun setup job first (addGroupCoverages by coarse_v7 and addMotifAnnotations).\n')
}

## 2. Setup metadata (if not already done)

In [ ]:
if (!has_coarse_v7_col) {
    clust_path <- paste0(projdir, 'csv/clusters/rna_leiden_v7.csv')
    clusters   <- read.csv(clust_path, sep = '\t', row.names = 1)

    split_strings     <- strsplit(rownames(clusters), '_')
    barcodes          <- sapply(split_strings, `[`, 1)
    sample            <- sapply(split_strings, `[`, 2)
    clusters$archr_bc <- paste0(sample, '#', barcodes)
    rownames(clusters) <- clusters$archr_bc

    proj <- addCellColData(proj, data = clusters$coarse_v7, name = 'coarse_v7', cells = rownames(clusters), force = TRUE)
}

meta <- as.data.frame(proj@cellColData)
cat('Cell counts per coarse_v7:\n')
print(table(meta$coarse_v7))

### load peaks

In [ ]:
peaks_path <- '/u/project/cluo/terencew/igvf/2023_YR2/multiome/atac/peaks/chromhmm/justin.chromhmm.dedup.bed'
peaks_df <- as.data.frame(fread(peaks_path))
peaks_df <- peaks_df[,1:3]
colnames(peaks_df) <- c('chrom', 'start', 'end')
peaks_granges <- makeGRangesFromDataFrame(peaks_df)
peaks_granges

In [ ]:
proj <- addPeakSet(
  ArchRProj = proj,
  peakSet = peaks_granges,
  genomeAnnotation = getGenomeAnnotation(proj),
  force = FALSE
)

### load motifs

In [ ]:
if (!'Motif' %in% names(proj@peakAnnotation)) {
    cat('Adding CISBP motif annotations...\n')
    proj <- addMotifAnnotations(
        ArchRProj  = proj,
        motifSet   = 'cisbp',
        name       = 'Motif',
        force      = FALSE
    )
} else {
    cat('Motif annotations already present.\n')
}

cat('\nMotif annotation names:', names(proj@peakAnnotation), '\n')

## 3. Get motif positions from peak set

Select TFs of interest based on SCENIC+ hub analysis and cluster-specific regulons:
- **Hub TFs** (high connectivity): NFE2L2, KLF6, ZBTB7A
- **Fail1-specific**: BNC2, ZEB1
- **Start-enriched**: CEBPB, CEBPD
- **Reprogramming factors**: KLF4, POU5F1 (OCT4), SOX2
- **Inter1-specific**: BCLAF1

In [ ]:
# TFs of interest — adjust based on what motifs are available in CISBP
# tfs_of_interest <- c(
#     'NFE2L2', 'KLF6', 'KLF4', 'ZBTB7A',
#     'BNC2', 'ZEB1',
#     'CEBPB', 'CEBPD',
#     'POU5F1', 'SOX2',
#     'BCLAF1', 'NFE2L1'
# )
tfs_of_interest <- c(
    'NFE2L2', 'KLF6', 'KLF4', 'ZBTB7A',
    'ZEB1',
    'CEBPB', 'CEBPD',
    'POU5F1', 'SOX2', 'MYC',
    'FOS', 'JUN',
    'NFE2L1'
)
# tfs_of_interest <- c('KLF4', 'SOX2', 'POU5F1', 'MYC', 'FOS', 'JUN')

# Pull motif positions from peak annotation
motif_positions <- getPositions(proj)
cat('Total motifs annotated:', length(motif_positions), '\n')

# Find which motif names match TFs of interest (CISBP names can have suffixes)
motif_names <- names(motif_positions)
matched <- sapply(tfs_of_interest, function(tf) {
    hits <- grep(paste0('^', tf, '_'), motif_names, value = TRUE, ignore.case = TRUE)
    if (length(hits) == 0) hits <- grep(tf, motif_names, value = TRUE, ignore.case = TRUE)
    hits[1]  # take first match
})
names(matched) <- tfs_of_interest
print(matched)

In [ ]:
# Subset to matched motifs (drop NAs)
matched_clean   <- matched[!is.na(matched)]
motif_positions_sub <- motif_positions[matched_clean]
names(motif_positions_sub) <- names(matched_clean)  # rename to clean TF names

cat('Motifs to footprint:', paste(names(motif_positions_sub), collapse = ', '), '\n')

## 4. Compute footprints per cluster

Uses pseudobulk insertion profiles (GroupCoverages) around motif binding sites.
Requires GroupCoverages by `coarse_v7`.

In [ ]:
proj <- addGroupCoverages(proj,
                          groupBy='coarse_v7')

In [ ]:
seFoot <- getFootprints(
    ArchRProj   = proj,
    positions   = motif_positions_sub,
    groupBy     = 'coarse_v7',
    flank       = 250,
    minCells    = 50,
    verbose     = TRUE
)
seFoot

In [ ]:
out <- paste0(projdir, '/rds/atac/processed/archr_footprint_intersect_rna.rds')
saveRDS(seFoot, out)

## 5. Plot footprints — all clusters

In [ ]:
plot_dir <- paste0(projdir, 'atac/footprints/')
dir.create(plot_dir, showWarnings = FALSE, recursive = TRUE)

plotFootprints(
    seFoot          = seFoot,
    ArchRProj       = proj,
    normMethod      = 'Subtract',
    plotName        = 'coarse_v7_footprints_subtract',
    addDOC          = FALSE,
    smoothWindow    = 5,
#     outputDirectory = plot_dir
)

## 6. Extract footprint scores per TF × cluster

In [ ]:
# seFoot is a SummarizedExperiment: rows = positions, cols = groups
# assayNames(seFoot) typically includes 'footprint' and 'norm' arrays
cat('Assay names:', assayNames(seFoot), '\n')
cat('Dimensions:', dim(seFoot), '\n')
cat('Groups:', head(colnames(seFoot)), '\n')

In [ ]:
compute_footprint_score <- function(seFoot, tf_name, assay_name = 'norm') {
    mat <- assay(seFoot[tf_name, ], assay_name)  # positions x groups
    positions <- rowData(seFoot)$position

    flank_mask   <- abs(positions) >= 100 & abs(positions) <= 250
    central_mask <- abs(positions) <= 30

    flank_mean   <- colMeans(mat[flank_mask, , drop = FALSE], na.rm = TRUE)
    central_mean <- colMeans(mat[central_mask, , drop = FALSE], na.rm = TRUE)

    score <- flank_mean - central_mean  # higher = deeper footprint
    data.frame(
        cluster         = names(score),
        footprint_score = as.numeric(score),
        tf              = tf_name
    )
}

score_list <- lapply(names(motif_positions_sub), compute_footprint_score, seFoot = seFoot)
score_df   <- do.call(rbind, score_list)

score_df$cluster <- factor(score_df$cluster, levels = clusters_order)

head(score_df)

In [ ]:
# Heatmap: footprint score, rows = TF, cols = cluster
ggplot(score_df, aes(x = cluster, y = tf, fill = footprint_score)) +
    geom_tile() +
    scale_fill_gradient2(low = 'blue', mid = 'white', high = 'red',
                         midpoint = 0, name = 'Footprint\nScore') +
    theme_bw() +
    theme(axis.text.x = element_text(angle = 45, hjust = 1)) +
    labs(title = 'TF footprint depth per cluster',
         x = 'Cluster', y = 'TF')

In [ ]:
# Rank TFs by footprint score within each cluster
score_df %>%
    arrange(cluster, desc(footprint_score)) %>%
    print(n = 50)

## 7. Export

In [ ]:
dir.create(out_dir, showWarnings = FALSE, recursive = TRUE)

fwrite(score_df, paste0(out_dir, 'footprint_scores_coarse_v7.csv'), sep = '\t')
saveRDS(seFoot,  paste0(out_dir, 'seFoot_coarse_v7.rds'))

cat('Outputs written to:', out_dir, '\n')

In [ ]:
sessionInfo()

In [ ]:
Sys.time()